# 6.2 常見片段怎麼合併？

# 第 6 章：自己的中英文 tokenizer

前置：字元ID與下一字loss。Tokenizer訓練是統計切詞規則，不是訓練LLM。早期與CLI使用可還原ByteTokenizer；BPE在本章獨立學習，換模型時要同步詞表與checkpoint。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：常見卡片合併：不是先懂詞義**

每次合併一對常見相鄰符號。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/bpe.svg")))

**先想一想：**合併a+b後，原序列長度會怎樣？

BPE每次合併最常見的一對相鄰符號。它學習的是文字壓縮規則，不知道詞義；訓練文案變了，合併次序也可能變。

**把直覺寫成數學：**同一對相鄰符號合併成新符號；不跨文件邊界統計pair。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from collections import Counter

corpus = [list("abab"), list("abac")]
pairs = Counter((a, b) for row in corpus for a, b in zip(row, row[1:]))
pair = pairs.most_common(1)[0][0]


def merge(row, pair):
    out = []
    i = 0
    while i < len(row):
        if i + 1 < len(row) and tuple(row[i : i + 2]) == pair:
            out.append("".join(pair))
            i += 2
        else:
            out.append(row[i])
            i += 1
    return out


print(pair, [merge(row, pair) for row in corpus])

**如何讀結果：**這是一個merge，不是完整生產tokenizer；只依training文字計數。

**只改一件事：**只增加一份含ac的文件，觀察最常見pair。
